# Week 4 Lab — Diagnostic Testing: Violations of the OLS Assumptions

**MANG2074 Financial Econometrics 1**

**Objectives**

- Test for heteroscedasticity (Breusch–Pagan, White) and apply robust (HC/HAC) standard errors.
- Test for autocorrelation (Durbin–Watson, Breusch–Godfrey).
- Test residual normality (Jarque–Bera) and handle outliers with impulse dummies.
- Diagnose multicollinearity (correlation matrix, VIF), functional form (RESET) and parameter stability (Chow test).

**Data**

- `../data/macro.csv` — the Week 3 APT-style model for Microsoft.
- `../data/capm.csv` — the Week 2 CAPM regression for Ford (Chow test).


## Setup — rebuild the Week 3 model

Run the cell below: it reconstructs the transformed dataset and re-estimates the APT-style regression whose residuals we will now put on trial.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.diagnostic import (het_breuschpagan, het_white,
                                          acorr_breusch_godfrey, linear_reset)
from statsmodels.stats.stattools import durbin_watson, jarque_bera
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# --- rebuild the Week 3 dataset and APT-style model ---
raw = pd.read_csv('../data/macro.csv', index_col=0, parse_dates=True)

def logdiff(x):
    return 100 * np.log(x / x.shift(1))

data = pd.DataFrame({
    'ermsoft':    logdiff(raw['MICROSOFT']) - raw['USTB3M'] / 12,
    'ersandp':    logdiff(raw['SANDP']) - raw['USTB3M'] / 12,
    'dprod':      raw['INDPRO'].diff(),
    'dcredit':    raw['CCREDIT'].diff(),
    'dinflation': logdiff(raw['CPI']).diff(),
    'dmoney':     raw['M1SUPPLY'].diff(),
    'dspread':    raw['BMINUSA'].diff(),
    'rterm':      (raw['USTB10Y'] - raw['USTB3M']).diff(),
}).dropna()

formula = 'ermsoft ~ ersandp + dprod + dcredit + dinflation + dmoney + dspread + rterm'
results = smf.ols(formula, data).fit()
print(results.summary().tables[1])

## Task 1 — Look before you test

Plot the residuals (`results.resid`) over time, and as a scatter against the fitted values (`results.fittedvalues`). Do you *see* heteroscedasticity, autocorrelation or outliers?

In [ ]:
resid = results.resid
# YOUR CODE HERE: time-series plot of resid, and scatter of resid vs results.fittedvalues


## Task 2 — Heteroscedasticity tests

Run the Breusch–Pagan and White tests. Both use the regression design matrix `results.model.exog`. Report the LM statistic and p-value for each and state your conclusion.

```python
lm, lm_p, f, f_p = het_breuschpagan(results.resid, results.model.exog)
```

In [ ]:
X = results.model.exog
# YOUR CODE HERE: het_breuschpagan and het_white; print LM stats and p-values


## Task 3 — Robust standard errors

Even if tests are borderline, robust SEs are cheap insurance. Re-fit the model with (a) White heteroscedasticity-robust SEs (`cov_type='HC1'`) and (b) Newey–West HAC SEs with 6 lags (`cov_type='HAC', cov_kwds={'maxlags': 6}`). Build a small table comparing the three sets of SEs. Do any conclusions change?

In [ ]:
res_hc = smf.ols(formula, data).fit(cov_type='HC1')
# YOUR CODE HERE: res_hac with Newey-West (maxlags=6); comparison DataFrame of bse


## Task 4 — Autocorrelation

(a) Compute the Durbin–Watson statistic from the residuals; recall $DW \approx 2(1-\hat\rho)$.
(b) Run the Breusch–Godfrey test with 12 lags: `acorr_breusch_godfrey(results, nlags=12)`. Why is BG more general than DW?

In [ ]:
# YOUR CODE HERE


## Task 5 — Non-normality and outlier dummies

(a) Jarque–Bera test on the residuals (`jarque_bera(results.resid)`).
(b) Find the months with the largest absolute residuals (`resid.abs().nlargest(5)`).
(c) Create impulse dummies for April 2000 and December 2000 (the dot-com months), re-estimate, and re-run JB. What does each dummy do, mechanically?

In [ ]:
# (a)
# YOUR CODE HERE

# (b)
# YOUR CODE HERE

# (c)
data['apr00'] = (data.index == '2000-04-01').astype(float)
# YOUR CODE HERE: dec00 dummy, re-estimate formula + ' + apr00 + dec00', JB again


## Task 6 — Multicollinearity

Compute the correlation matrix of the seven factors and the VIF for each (use `variance_inflation_factor` on `sm.add_constant(X_df).values`, skipping the constant). Is multicollinearity a problem here?

In [ ]:
factors = ['ersandp', 'dprod', 'dcredit', 'dinflation', 'dmoney', 'dspread', 'rterm']
# YOUR CODE HERE: correlation matrix and VIFs


## Task 7 — Functional form: Ramsey RESET

Run `linear_reset(results, power=3, use_f=True)`. State the null hypothesis and your conclusion.

In [ ]:
# YOUR CODE HERE


## Task 8 — Structural stability: Chow test on the Ford CAPM

Rebuild the Ford CAPM regression (Week 2, `capm.csv`). Test for a structural break at **January 2008** (the financial crisis): estimate the model on the full sample, pre-2008 and post-2008 samples, then compute

$$F = \frac{(RSS_p - (RSS_1+RSS_2))/k}{(RSS_1+RSS_2)/(T-2k)}$$

with $k=2$. Compare with $F(k, T-2k)$ critical values. Has Ford's beta changed?

In [ ]:
capm = pd.read_csv('../data/capm.csv', index_col=0, parse_dates=True)
capm['erford'] = 100 * np.log(capm['FORD']).diff() - capm['USTB3M'] / 12
capm['ersandp'] = 100 * np.log(capm['SANDP']).diff() - capm['USTB3M'] / 12
capm = capm.dropna()

# YOUR CODE HERE: pooled, pre ('2002':'2007'), post ('2008':) regressions and the Chow F


## Task 9 — Diagnostic report

Summarise your audit of the APT model in a short table or bullet list: for each assumption, the test, the verdict, and the action taken (if any). 6–10 lines.

*YOUR ANSWER HERE*